# FBCCA–DANN embedding–KNN：SSVEP 无监督域适配

本 notebook 严格采用以下结构：

```text
EEG → 固定FBCCA特征 → 可训练embedding编码器 → KNN频率分类
                         ├─ source监督对比损失
                         └─ GRL + source/target域损失
```

- **KNN是唯一的频率分类器**，输出15/12/10 Hz。
- 网络中不存在频率分类头，不使用频率交叉熵。
- FBCCA没有可训练参数。
- `ADAPTATION_FILES`只加载EEG，不读取频率标签。
- `DRIFT_FILES`在编码器、scaler和KNN全部确定后才加载，只用于最终测试。
- 使用目标session无标签EEG进行适配，因此实验名称是UDA，不是zero-shot。


In [1]:
from functools import lru_cache
from pathlib import Path
import itertools
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import cheb1ord, cheby1, filtfilt
from sklearn.cross_decomposition import CCA
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## 1. 文件角色与不可泄漏约束

修改测试文件时只调整下面四个文件列表。代码会打印角色并检查各列表互不重叠。

验证集只从`TRAIN_FILES`内部按“文件 × 频率类别”分层切分。`TEST_FILES`和`DRIFT_FILES`不参与训练、早停或超参数选择。


In [2]:
DATA_DIR = Path("data")

TRAIN_FILES = [f"test_{index}_ssvep" for index in range(2, 8)]
TEST_FILES = ["test_8_ssvep"]
ADAPTATION_FILES = [f"four_{index}_ssvep" for index in (1, 2, 3, 4)]
DRIFT_FILES = ["four_5_ssvep"]

VALIDATION_FRACTION = 0.20
FREQUENCIES_HZ = np.array([15.0, 12.0, 10.0])
SAMPLE_RATE_HZ = 250.0
FOUR_DELAY_SECONDS = 0.5
FOUR_WINDOW_SECONDS = 1.5
NUM_HARMONICS = 4
NUM_FILTER_BANKS = 5

PASSBAND_HZ = np.array([6, 14, 22, 30, 38, 46, 54, 62, 70, 78], dtype=float)
STOPBAND_HZ = np.array([4, 10, 16, 24, 32, 40, 48, 56, 64, 72], dtype=float)
FILTER_WEIGHTS = np.array([
    index ** (-1.25) + 0.25 for index in range(1, NUM_FILTER_BANKS + 1)
])

DANN_BATCH_SIZE = 64
DANN_MAX_EPOCHS = 300
DANN_PATIENCE = 45
DANN_LEARNING_RATE = 1e-3
DANN_WEIGHT_DECAY = 1e-4
DANN_DOMAIN_WEIGHT = 0.10
DANN_TEMPERATURE = 0.10
EMBEDDING_DIM = 32


def title(file_names):
    return "/".join(name.removesuffix("_ssvep") for name in file_names)


all_roles = TRAIN_FILES + TEST_FILES + ADAPTATION_FILES + DRIFT_FILES
assert len(all_roles) == len(set(all_roles)), "file leakage: one file appears in multiple roles"
assert TRAIN_FILES and TEST_FILES and ADAPTATION_FILES and DRIFT_FILES
assert all(name.startswith("test_") for name in TRAIN_FILES + TEST_FILES)
assert all(name.startswith("four_") for name in ADAPTATION_FILES + DRIFT_FILES)

print("file roles")
print("  labeled source train/validation:", title(TRAIN_FILES))
print("  held-out source test:", title(TEST_FILES))
print("  unlabeled target adaptation:", title(ADAPTATION_FILES))
print("  final target test:", title(DRIFT_FILES))


file roles
  labeled source train/validation: test_2/test_3/test_4/test_5/test_6/test_7
  held-out source test: test_8
  unlabeled target adaptation: four_1/four_2/four_3/four_4
  final target test: four_5


In [3]:
def crop_trial_array(x, name):
    x = np.stack(x) if np.asarray(x).dtype == object else np.asarray(x)
    if x.ndim != 3:
        raise ValueError(f"{name}: expected [trial, channel, time], got {x.shape}")
    if name.startswith("four_"):
        start = round(FOUR_DELAY_SECONDS * SAMPLE_RATE_HZ)
        requested = round(FOUR_WINDOW_SECONDS * SAMPLE_RATE_HZ)
        if x.shape[-1] <= start:
            raise ValueError(f"{name}: cannot skip {start} samples from {x.shape[-1]}")
        x = x[:, :, start:min(start + requested, x.shape[-1])]
    elif not name.startswith("test_"):
        raise ValueError(f"{name}: unknown session prefix")
    return x.astype(np.float64)


def load_labeled_files(file_names):
    records = []
    for name in file_names:
        x = crop_trial_array(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)
        if len(x) != len(y):
            raise ValueError(f"{name}: trial/label length mismatch")
        if not np.isin(y, np.arange(len(FREQUENCIES_HZ))).all():
            raise ValueError(f"{name}: labels must be 0..{len(FREQUENCIES_HZ) - 1}")
        records.append((x, y, name))
        print("labeled", name, x.shape, np.bincount(y, minlength=len(FREQUENCIES_HZ)))
    return records


def load_unlabeled_files(file_names):
    # Deliberately constructs no label path and never calls np.load for target labels.
    records = []
    for name in file_names:
        x = crop_trial_array(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        records.append((x, name))
        print("unlabeled", name, x.shape)
    return records


def concatenate_labeled(records):
    trials = [trial for x, _, _ in records for trial in x]
    result = np.empty(len(trials), dtype=object)
    result[:] = trials
    labels = np.concatenate([y for _, y, _ in records])
    sources = np.concatenate([
        np.full(len(y), name, dtype=object) for _, y, name in records
    ])
    return result, labels, sources


def concatenate_unlabeled(records):
    trials = [trial for x, _ in records for trial in x]
    result = np.empty(len(trials), dtype=object)
    result[:] = trials
    return result


# Before training, load only labeled source training files and unlabeled adaptation EEG.
source_records = load_labeled_files(TRAIN_FILES)
adaptation_records = load_unlabeled_files(ADAPTATION_FILES)
X_pool, y_pool, source_pool = concatenate_labeled(source_records)
X_adaptation = concatenate_unlabeled(adaptation_records)

pool_indices = np.arange(len(y_pool))
strata = np.array([f"{source}::{label}" for source, label in zip(source_pool, y_pool)])
train_indices, validation_indices = train_test_split(
    pool_indices,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=strata,
)
assert set(train_indices).isdisjoint(validation_indices)
X_train, y_train = X_pool[train_indices], y_pool[train_indices]
X_validation, y_validation = X_pool[validation_indices], y_pool[validation_indices]
print("source train / validation:", len(y_train), len(y_validation))
print("unlabeled adaptation trials:", len(X_adaptation))


labeled test_2_ssvep (19, 8, 374) [8 4 7]
labeled test_3_ssvep (41, 8, 373) [11 16 14]
labeled test_4_ssvep (69, 8, 374) [17 23 29]
labeled test_5_ssvep (102, 8, 373) [30 35 37]
labeled test_6_ssvep (106, 8, 373) [17 43 46]
labeled test_7_ssvep (67, 8, 373) [19 20 28]
unlabeled four_1_ssvep (46, 8, 373)
unlabeled four_2_ssvep (152, 8, 372)
unlabeled four_3_ssvep (95, 8, 372)
unlabeled four_4_ssvep (122, 8, 372)
source train / validation: 323 81
unlabeled adaptation trials: 415


## 2. 固定FBCCA特征

每个trial逐通道去均值并按标准差归一化。使用5个滤波器组、4个谐波和3个目标频率。每个trial产生22维固定特征：15维滤波器组相关分数、3维总分、3维归一化总分和1维第一/第二名差值。


In [4]:
def trial_zscore(x):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / np.maximum(std, 1e-8)


@lru_cache(maxsize=None)
def filter_coefficients(bank_index):
    nyquist = SAMPLE_RATE_HZ / 2.0
    wp = [PASSBAND_HZ[bank_index - 1] / nyquist, 90.0 / nyquist]
    ws = [STOPBAND_HZ[bank_index - 1] / nyquist, 100.0 / nyquist]
    order, wn = cheb1ord(wp, ws, 3, 40)
    return cheby1(order, 0.5, wn, btype="bandpass")


def filter_bank(eeg, bank_index):
    b, a = filter_coefficients(bank_index)
    return filtfilt(b, a, eeg, axis=-1)


@lru_cache(maxsize=None)
def reference_signals(frequency, sample_count):
    time = np.arange(sample_count) / SAMPLE_RATE_HZ
    waves = []
    for harmonic in range(1, NUM_HARMONICS + 1):
        waves.extend([
            np.sin(2 * np.pi * harmonic * frequency * time),
            np.cos(2 * np.pi * harmonic * frequency * time),
        ])
    return np.asarray(waves).T


def canonical_correlation(eeg, reference):
    usable = eeg[np.std(eeg, axis=1) > 1e-8]
    if len(usable) == 0:
        return 0.0
    cca = CCA(n_components=1, max_iter=1000)
    eeg_component, reference_component = cca.fit_transform(usable.T, reference)
    value = np.corrcoef(eeg_component[:, 0], reference_component[:, 0])[0, 1]
    return 0.0 if not np.isfinite(value) else float(value)


def fbcca_features(trial):
    trial = trial_zscore(np.asarray(trial, dtype=np.float64))
    filtered = [filter_bank(trial, bank) for bank in range(1, NUM_FILTER_BANKS + 1)]
    weighted = np.zeros((len(FREQUENCIES_HZ), NUM_FILTER_BANKS), dtype=np.float64)
    for frequency_index, frequency in enumerate(FREQUENCIES_HZ):
        reference = reference_signals(float(frequency), trial.shape[-1])
        for bank_index, bank_eeg in enumerate(filtered):
            rho = canonical_correlation(bank_eeg, reference)
            weighted[frequency_index, bank_index] = FILTER_WEIGHTS[bank_index] * rho**2
    scores = weighted.sum(axis=1)
    normalized = scores / max(scores.sum(), 1e-12)
    ordered = np.sort(scores)
    margin = ordered[-1] - ordered[-2]
    features = np.concatenate([weighted.ravel(), scores, normalized, [margin]])
    return features.astype(np.float32), scores.astype(np.float32)


def transform_trials(trials, name):
    features, scores = [], []
    for index, trial in enumerate(trials, start=1):
        feature, score = fbcca_features(trial)
        features.append(feature)
        scores.append(score)
        if index % 50 == 0 or index == len(trials):
            print(f"{name}: {index}/{len(trials)}")
    return np.asarray(features), np.asarray(scores)


features_train, scores_train = transform_trials(X_train, "source train")
features_validation, scores_validation = transform_trials(X_validation, "source validation")
features_adaptation, _ = transform_trials(X_adaptation, "unlabeled target adaptation")
print("FBCCA feature dimension:", features_train.shape[1])


source train: 50/323
source train: 100/323
source train: 150/323
source train: 200/323
source train: 250/323
source train: 300/323
source train: 323/323
source validation: 50/81
source validation: 81/81
unlabeled target adaptation: 50/415
unlabeled target adaptation: 100/415
unlabeled target adaptation: 150/415
unlabeled target adaptation: 200/415
unlabeled target adaptation: 250/415
unlabeled target adaptation: 300/415
unlabeled target adaptation: 350/415
unlabeled target adaptation: 400/415
unlabeled target adaptation: 415/415
FBCCA feature dimension: 22


## 3. 只训练embedding与二分类域头

编码器通过两种损失训练：source频率标签的监督对比损失保持类别结构；GRL域损失降低source/target差异。网络中没有频率分类头。

早停只看source验证集上的固定5-NN准确率。scaler只在source训练特征上拟合。


In [5]:
scaler = RobustScaler().fit(features_train)
train_scaled = scaler.transform(features_train).astype(np.float32)
validation_scaled = scaler.transform(features_validation).astype(np.float32)
adaptation_scaled = scaler.transform(features_adaptation).astype(np.float32)

domain_features = np.concatenate([train_scaled, adaptation_scaled])
frequency_labels = np.concatenate([
    y_train,
    np.full(len(adaptation_scaled), -1, dtype=np.int64),
])
domain_labels = np.concatenate([
    np.zeros(len(train_scaled), dtype=np.int64),
    np.ones(len(adaptation_scaled), dtype=np.int64),
])
domain_counts = np.bincount(domain_labels)
sample_weights = 1.0 / domain_counts[domain_labels]
sampler = WeightedRandomSampler(
    torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(domain_features),
        torch.tensor(frequency_labels, dtype=torch.long),
        torch.tensor(domain_labels, dtype=torch.long),
    ),
    batch_size=DANN_BATCH_SIZE,
    sampler=sampler,
)


class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, strength):
        ctx.strength = strength
        return x.view_as(x)

    @staticmethod
    def backward(ctx, gradient):
        return -ctx.strength * gradient, None


def grl(x, strength):
    return GradientReversal.apply(x, strength)


class DomainAdversarialEmbedding(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.ELU(),
            nn.Dropout(0.20),
            nn.Linear(64, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.ELU(),
        )
        self.domain_head = nn.Sequential(
            nn.Linear(embedding_dim, 32),
            nn.ELU(),
            nn.Linear(32, 2),
        )

    def forward(self, x, grl_strength=0.0):
        embedding = self.encoder(x)
        domain_logits = self.domain_head(grl(embedding, grl_strength))
        return embedding, domain_logits


def supervised_contrastive_loss(embedding, labels, temperature):
    embedding = F.normalize(embedding, dim=1)
    logits = embedding @ embedding.T / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    self_mask = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    positive_mask = labels[:, None].eq(labels[None, :]) & ~self_mask
    exp_logits = torch.exp(logits) * ~self_mask
    log_probability = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    positive_count = positive_mask.sum(dim=1)
    valid = positive_count > 0
    if not valid.any():
        return embedding.sum() * 0.0
    mean_log_probability = (
        (positive_mask * log_probability).sum(dim=1) / positive_count.clamp_min(1)
    )
    return -mean_log_probability[valid].mean()


model = DomainAdversarialEmbedding(train_scaled.shape[1], EMBEDDING_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=DANN_LEARNING_RATE, weight_decay=DANN_WEIGHT_DECAY
)


@torch.no_grad()
def encode(features):
    model.eval()
    tensor = torch.tensor(features, dtype=torch.float32, device=DEVICE)
    return model(tensor, 0.0)[0].cpu().numpy()


def validation_knn_accuracy():
    train_embedding = encode(train_scaled)
    validation_embedding = encode(validation_scaled)
    provisional_knn = KNeighborsClassifier(n_neighbors=5, weights="distance", p=2)
    provisional_knn.fit(train_embedding, y_train)
    return provisional_knn.score(validation_embedding, y_validation)


best_validation_accuracy = -1.0
best_state = None
epochs_without_improvement = 0
history = {"contrastive_loss": [], "domain_balanced_accuracy": [], "validation_knn": []}

for epoch in range(1, DANN_MAX_EPOCHS + 1):
    model.train()
    progress = (epoch - 1) / max(DANN_MAX_EPOCHS - 1, 1)
    grl_strength = float(2 / (1 + np.exp(-10 * progress)) - 1)
    contrastive_sum = 0.0
    batch_count = 0
    epoch_domain_true, epoch_domain_prediction = [], []

    for batch_features, batch_frequencies, batch_domains in train_loader:
        batch_features = batch_features.to(DEVICE)
        batch_frequencies = batch_frequencies.to(DEVICE)
        batch_domains = batch_domains.to(DEVICE)
        embedding, domain_logits = model(batch_features, grl_strength)
        labeled = batch_frequencies >= 0
        contrastive_loss = supervised_contrastive_loss(
            embedding[labeled], batch_frequencies[labeled], DANN_TEMPERATURE
        )
        domain_loss = F.cross_entropy(domain_logits, batch_domains)
        loss = contrastive_loss + DANN_DOMAIN_WEIGHT * domain_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        contrastive_sum += contrastive_loss.item()
        batch_count += 1
        epoch_domain_true.extend(batch_domains.cpu().tolist())
        epoch_domain_prediction.extend(domain_logits.argmax(1).detach().cpu().tolist())

    mean_contrastive = contrastive_sum / max(batch_count, 1)
    domain_balanced = balanced_accuracy_score(epoch_domain_true, epoch_domain_prediction)
    validation_accuracy = validation_knn_accuracy()
    history["contrastive_loss"].append(mean_contrastive)
    history["domain_balanced_accuracy"].append(domain_balanced)
    history["validation_knn"].append(validation_accuracy)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"epoch={epoch:03d} supcon={mean_contrastive:.4f} "
            f"domain_bal_acc={domain_balanced:.4f} "
            f"source_val_5nn={validation_accuracy:.4f} grl={grl_strength:.3f}"
        )
    if epochs_without_improvement >= DANN_PATIENCE:
        print("early stop at epoch", epoch)
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()
print("best source validation 5-NN accuracy:", best_validation_accuracy)
print("network frequency heads:", 0)


epoch=001 supcon=3.6330 domain_bal_acc=0.5466 source_val_5nn=1.0000 grl=0.000
epoch=020 supcon=2.4028 domain_bal_acc=0.7037 source_val_5nn=0.9753 grl=0.307
epoch=040 supcon=2.4603 domain_bal_acc=0.5907 source_val_5nn=0.9753 grl=0.573
early stop at epoch 46
best source validation 5-NN accuracy: 1.0
network frequency heads: 0


## 4. 仅用source验证集选择并拟合最终KNN

编码器冻结后，KNN超参数只根据source验证集选择。最终KNN仅拟合source训练embedding和频率标签。


In [6]:
embedding_train = encode(train_scaled)
embedding_validation = encode(validation_scaled)

candidates = []
for neighbors, weights, p in itertools.product(
    [1, 3, 5, 7, 9, 11, 15, 21], ["uniform", "distance"], [1, 2]
):
    candidate = KNeighborsClassifier(n_neighbors=neighbors, weights=weights, p=p)
    candidate.fit(embedding_train, y_train)
    accuracy = candidate.score(embedding_validation, y_validation)
    candidates.append((accuracy, -neighbors, weights == "distance", -p, candidate))

best_knn_validation, _, _, _, knn = max(candidates, key=lambda item: item[:4])
print("selected final KNN:", knn)
print("source validation KNN accuracy:", best_knn_validation)


selected final KNN: KNeighborsClassifier(n_neighbors=3, weights='distance')
source validation KNN accuracy: 1.0


## 5. 模型冻结后加载最终测试文件

执行到这里，FBCCA定义、scaler、embedding编码器和KNN均已确定。现在才加载`TEST_FILES`和`DRIFT_FILES`。两个测试集都不会反馈到任何训练或模型选择步骤。


In [7]:
source_test_records = load_labeled_files(TEST_FILES)
drift_test_records = load_labeled_files(DRIFT_FILES)
X_source_test, y_source_test, _ = concatenate_labeled(source_test_records)
X_drift_test, y_drift_test, _ = concatenate_labeled(drift_test_records)

features_source_test, scores_source_test = transform_trials(X_source_test, title(TEST_FILES))
features_drift_test, scores_drift_test = transform_trials(X_drift_test, title(DRIFT_FILES))

source_test_scaled = scaler.transform(features_source_test).astype(np.float32)
drift_test_scaled = scaler.transform(features_drift_test).astype(np.float32)
embedding_source_test = encode(source_test_scaled)
embedding_drift_test = encode(drift_test_scaled)


def report(name, labels, prediction):
    correct = int(np.sum(prediction == labels))
    accuracy = correct / len(labels)
    balanced = balanced_accuracy_score(labels, prediction)
    recalls = recall_score(
        labels, prediction, labels=np.arange(len(FREQUENCIES_HZ)), average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(labels, prediction, labels=np.arange(len(FREQUENCIES_HZ)))
    print(f"\n{name}")
    print(f"accuracy: {accuracy:.4f} ({correct}/{len(labels)})")
    print(f"balanced accuracy: {balanced:.4f}")
    for frequency, value in zip(FREQUENCIES_HZ, recalls):
        print(f"recall {frequency:g} Hz: {value:.4f}")
    print("confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]")
    print(matrix)
    return accuracy


source_fbcca_prediction = scores_source_test.argmax(axis=1)
source_knn_prediction = knn.predict(embedding_source_test)
drift_fbcca_prediction = scores_drift_test.argmax(axis=1)
drift_knn_prediction = knn.predict(embedding_drift_test)

report(f"{title(TEST_FILES)} pure FBCCA", y_source_test, source_fbcca_prediction)
report(f"{title(TEST_FILES)} DANN-embedding-KNN", y_source_test, source_knn_prediction)
fbcca_drift_accuracy = report(
    f"{title(DRIFT_FILES)} pure FBCCA", y_drift_test, drift_fbcca_prediction
)
knn_drift_accuracy = report(
    f"{title(DRIFT_FILES)} DANN-embedding-KNN", y_drift_test, drift_knn_prediction
)
print(f"\n{title(DRIFT_FILES)} accuracy change: {knn_drift_accuracy - fbcca_drift_accuracy:+.4f}")


labeled test_8_ssvep (67, 8, 373) [23 24 20]
labeled four_5_ssvep (125, 8, 372) [25 44 56]
test_8: 50/67
test_8: 67/67
four_5: 50/125
four_5: 100/125
four_5: 125/125

test_8 pure FBCCA
accuracy: 0.9851 (66/67)
balanced accuracy: 0.9861
recall 15 Hz: 1.0000
recall 12 Hz: 0.9583
recall 10 Hz: 1.0000
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[23  0  0]
 [ 0 23  1]
 [ 0  0 20]]

test_8 DANN-embedding-KNN
accuracy: 0.9403 (63/67)
balanced accuracy: 0.9417
recall 15 Hz: 1.0000
recall 12 Hz: 0.8750
recall 10 Hz: 0.9500
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[23  0  0]
 [ 0 21  3]
 [ 0  1 19]]

four_5 pure FBCCA
accuracy: 0.8400 (105/125)
balanced accuracy: 0.8426
recall 15 Hz: 0.8800
recall 12 Hz: 0.7727
recall 10 Hz: 0.8750
confusion matrix, rows=true [15,12,10], columns=predicted [15,12,10]
[[22  0  3]
 [ 3 34  7]
 [ 5  2 49]]

four_5 DANN-embedding-KNN
accuracy: 0.8560 (107/125)
balanced accuracy: 0.8158
recall 15 Hz: 0.6000
recall

## 6. 报告规则

- 正式频率分类结果只有纯FBCCA和DANN-embedding-KNN。
- 不存在也不得报告“DANN频率分类头准确率”。
- 同时报告正确数/总数、每类召回率、balanced accuracy和混淆矩阵。
- 结果变差也原样保留，不根据最终测试结果调整模型、窗口、随机种子或标签映射。
- 若更换`DRIFT_FILES`，必须重新从头运行，并以实际打印的文件角色和样本数为准。
